# SigWavNet → Music Genre ClassificationFine-tuning **SigWavNet** (Nfissi et al., *IEEE T-AFFC* 2025, [arXiv:2502.00310](https://arxiv.org/abs/2502.00310))from speech emotion recognition to **GTZAN** music genre classification.**Runtime → Change runtime type → T4 GPU** before running anything.| Stage | Data | Purpose ||---|---|---|| A | RAVDESS (1440 clips, 8 emotions, speaker-independent split) | pretrain the learnable-FDWT front end + encoders on speech || B1 | GTZAN | new 10-class head only, backbone frozen || B2 | GTZAN | full network, discriminative LRs || Baseline | GTZAN | identical model trained from random init — the control |Original code: BSD 3-Clause, © 2024 Alaa Nfissi. See `third_party/LICENSE-SigWavNet`.

## 0. Environment

In [ ]:
!nvidia-smi -Limport torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
!pip -q install PyWavelets torchaudio soundfile kagglehub scikit-learn --upgrade

## 1. Get the codeReplace `YOUR_GITHUB_USERNAME` with yours once you've pushed the repo (see `README.md`).

In [ ]:
REPO = "https://github.com/YOUR_GITHUB_USERNAME/sigwavnet-music-genre.git"import os, sys, subprocessif not os.path.isdir("/content/sigwavnet-music-genre"):    subprocess.run(["git", "clone", REPO, "/content/sigwavnet-music-genre"])sys.path.insert(0, "/content/sigwavnet-music-genre/src")os.chdir("/content/sigwavnet-music-genre")!ls src

## 2. Datasets**GTZAN** — Kaggle mirror. Run `kagglehub.login()` once and paste your Kaggle username +API key (kaggle.com → Settings → Create New Token → open `kaggle.json`).**RAVDESS** — direct download from Zenodo, no auth (~200 MB).

In [ ]:
import kagglehub, os, globkagglehub.login()GTZAN_ROOT = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")GTZAN_ROOT = os.path.join(GTZAN_ROOT, "Data")print(GTZAN_ROOT, os.listdir(GTZAN_ROOT))

In [ ]:
import osRAVDESS_ROOT = "/content/ravdess"if not os.path.isdir(RAVDESS_ROOT):    !wget -q --show-progress https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip -O /content/rav.zip    !mkdir -p {RAVDESS_ROOT} && unzip -q /content/rav.zip -d {RAVDESS_ROOT}print(len(os.listdir(RAVDESS_ROOT)), "actor folders")

## 3. Index, cache, splitCaching resamples everything to 16 kHz mono float32 `.npy` once (~5 min for both sets).Splits are **by clip** for GTZAN and **by actor** for RAVDESS, so no window of a track andno utterance of a speaker appears in two splits.

In [ ]:
from data import (index_gtzan, index_ravdess, build_cache, split_by_group,                  SegmentDataset, class_weights, GTZAN_GENRES)rav = build_cache(index_ravdess(RAVDESS_ROOT), "/content/cache/ravdess")gtz = build_cache(index_gtzan(GTZAN_ROOT),     "/content/cache/gtzan")print("RAVDESS", rav.shape, "| GTZAN", gtz.shape)print(gtz.label.value_counts().to_dict())

In [ ]:
rav_tr, rav_va, rav_te = split_by_group(rav, test_size=0.15, val_size=0.15, seed=42)gtz_tr, gtz_va, gtz_te = split_by_group(gtz, test_size=0.15, val_size=0.15, seed=42)print(f"RAVDESS clips  train/val/test: {len(rav_tr)}/{len(rav_va)}/{len(rav_te)}")print(f"GTZAN   clips  train/val/test: {len(gtz_tr)}/{len(gtz_va)}/{len(gtz_te)}")RAV_CLASSES = sorted(rav.label.unique())SEG_SEC, HOP_SEC = 3.0, 1.5

In [ ]:
rav_tr_ds = SegmentDataset(rav_tr, RAV_CLASSES, SEG_SEC, HOP_SEC, train=True)rav_va_ds = SegmentDataset(rav_va, RAV_CLASSES, SEG_SEC, HOP_SEC, train=False)gtz_tr_ds = SegmentDataset(gtz_tr, GTZAN_GENRES, SEG_SEC, HOP_SEC, train=True)gtz_va_ds = SegmentDataset(gtz_va, GTZAN_GENRES, SEG_SEC, HOP_SEC, train=False)gtz_te_ds = SegmentDataset(gtz_te, GTZAN_GENRES, SEG_SEC, HOP_SEC, train=False)print("GTZAN eval windows:", len(gtz_va_ds), len(gtz_te_ds))

## 4. Stage A — pretrain on RAVDESS (speech emotion)`level=8`, `kernelInit=db10`, `PerFilter` kernels: the configuration the paper reports asoptimal. `n_channel` is dropped from 128 to 32 to fit a T4 — note this in your slides.

In [ ]:
from train import build_model, train_modelimport torch, timeCFG = dict(level=8, hidden_dim=64, n_layers=3, n_channel=32,           kernel="db10", mode="PerFilter", initHT=1.0, alpha=10.0, dropout=0.1)model_a = build_model(n_output=len(RAV_CLASSES), **CFG)print(sum(p.numel() for p in model_a.parameters() if p.requires_grad), "trainable params")t0 = time.time()model_a, hist_a = train_model(    model_a, rav_tr_ds, rav_va_ds, classes=RAV_CLASSES,    weights=class_weights(rav_tr, RAV_CLASSES),    epochs=25, batch_size=16,    lr_head=1e-3, lr_encoder=5e-4, lr_wavelet=2e-4,    freeze_epochs=0, ckpt_path="stageA_ravdess.pt")print("stage A took", round((time.time()-t0)/60, 1), "min")

## 5. Stage B — fine-tune on GTZAN`replace_head(10)` swaps the class-projection conv. `freeze_epochs=3` trains that headalone first so its random gradients don't wreck the pretrained wavelet filters, thenunfreezes everything with head > encoder > wavelet learning rates.

In [ ]:
model_b = build_model(n_output=len(RAV_CLASSES), **CFG)model_b.load_pretrained("stageA_ravdess.pt")model_b.replace_head(len(GTZAN_GENRES))model_b, hist_b = train_model(    model_b, gtz_tr_ds, gtz_va_ds, classes=GTZAN_GENRES,    weights=class_weights(gtz_tr, GTZAN_GENRES),    epochs=30, batch_size=16,    lr_head=1e-3, lr_encoder=2e-4, lr_wavelet=5e-5,    freeze_epochs=3, ckpt_path="stageB_finetuned.pt")

## 6. Baseline — same architecture, random initWithout this control there is no evidence that the transfer did anything.

In [ ]:
model_s = build_model(n_output=len(GTZAN_GENRES), **CFG)model_s, hist_s = train_model(    model_s, gtz_tr_ds, gtz_va_ds, classes=GTZAN_GENRES,    weights=class_weights(gtz_tr, GTZAN_GENRES),    epochs=30, batch_size=16,    lr_head=1e-3, lr_encoder=5e-4, lr_wavelet=2e-4,    freeze_epochs=0, ckpt_path="scratch_gtzan.pt")

## 7. Test-set evaluationClip-level: log-probabilities are averaged over every 3 s window of a track before argmax.**Report the clip-level numbers.** Segment-level scores are over correlated windows andaren't comparable to published GTZAN results.

In [ ]:
from evaluate import evaluate_clips, plot_confusion, plot_kernels, plot_history, save_resultsres_ft = evaluate_clips(model_b, gtz_te_ds, GTZAN_GENRES)res_sc = evaluate_clips(model_s, gtz_te_ds, GTZAN_GENRES)for tag, r in [("fine-tuned", res_ft), ("scratch", res_sc)]:    print(f"\n=== {tag} ===")    print(f"segment acc {r['segment_acc']:.4f}  f1 {r['segment_f1']:.4f}")    print(f"clip    acc {r['clip_acc']:.4f}  f1 {r['clip_f1']:.4f}")    print(r["report"])save_results("finetuned", res_ft); save_results("scratch", res_sc)

In [ ]:
plot_confusion(res_ft["cm"], GTZAN_GENRES, "confusion_finetuned.png",               title="GTZAN test — SigWavNet fine-tuned from RAVDESS")plot_confusion(res_sc["cm"], GTZAN_GENRES, "confusion_scratch.png",               title="GTZAN test — SigWavNet from scratch")plot_history({"fine-tuned": hist_b, "scratch": hist_s}, "training_curves.png")plot_kernels(model_b, "learned_kernels.png", n_levels=3)from IPython.display import Image, displayfor f in ["confusion_finetuned.png", "confusion_scratch.png",          "training_curves.png", "learned_kernels.png"]:    display(Image(f))

## 8. Ablations (optional, ~15 min each)Each row is one bar on your ablation slide. Run whichever you have time for.

In [ ]:
ABLATIONS = {  "no-LAHT (initHT=0, frozen)": dict(CFG, initHT=0.0),  "CQF kernels (shared)":       dict(CFG, mode="CQF"),  "frozen db10 (no learning)":  dict(CFG),          # + kernTrainable=False below  "level=4":                    dict(CFG, level=4),}results_abl = {}for name, cfg in ABLATIONS.items():    print("\n####", name)    m = build_model(n_output=len(GTZAN_GENRES), **cfg)    if "frozen db10" in name:        for k in list(m.kernelsG_) + list(m.kernelsH_):            k.kernel.requires_grad_(False)    m, _ = train_model(m, gtz_tr_ds, gtz_va_ds, classes=GTZAN_GENRES,                       weights=class_weights(gtz_tr, GTZAN_GENRES),                       epochs=15, batch_size=16, ckpt_path=f"abl_{abs(hash(name))}.pt",                       log_every=0)    r = evaluate_clips(m, gtz_te_ds, GTZAN_GENRES)    results_abl[name] = r["clip_acc"]    print(name, "->", round(r["clip_acc"], 4))    del m; torch.cuda.empty_cache()print(results_abl)

## 9. Save artefactsDownload `results.json` and the PNGs, then paste the numbers into `make_deck.js`(`RESULTS` at the top) and rebuild the slides.

In [ ]:
!zip -q -r artefacts.zip results.json *.png *_history.json stageB_finetuned.pt scratch_gtzan.ptfrom google.colab import files; files.download("artefacts.zip")